# ChromFound LoRA Fine-tuning Tutorial

This notebook demonstrates how to use the Low-Rank Adaptation (LoRA) fine-tuning capabilities of ChromFound.

In [ ]:
import torch
import sys
import os

# Add the src directory to the path
sys.path.append(os.path.join(os.getcwd(), "src"))

## Understanding LoRA in ChromFound

LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that adds low-rank decomposition matrices to the attention layers during fine-tuning. This approach:

- Significantly reduces the number of trainable parameters
- Decreases memory usage
- Helps prevent catastrophic forgetting of pre-trained representations
- Enables efficient fine-tuning on multiple downstream tasks

In [ ]:
# Import the necessary modules
from src.models.finetune_model import create_finetune_model_from_pretrained, count_parameters

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Creating a Fine-tuned Model with LoRA

Let's create a model with LoRA for fine-tuning on a downstream task:

In [ ]:
# Define paths to the pretrained model
checkpoint_dir = "src/checkpoints"  # Update this path to your checkpoint directory

# Create the fine-tuning model with LoRA
model = create_finetune_model_from_pretrained(
    checkpoint_dir=checkpoint_dir,
    lora_rank=8,           # Rank of the low-rank adaptation
    lora_alpha=16.0,       # Scaling factor for LoRA
    lora_dropout=0.0,      # Dropout rate for LoRA layers
    task_type="classification",  # Task type
    freeze_backbone=True   # Freeze the original model parameters
)

# Move model to device
model = model.to(device)

# Count parameters to see the efficiency of LoRA
param_counts = count_parameters(model)
print(f"Total parameters: {param_counts['total_parameters']:,}")
print(f"Trainable parameters: {param_counts['trainable_parameters']:,}")
print(f"LoRA parameters: {param_counts['lora_parameters']:,}")
print(f"Frozen parameters: {param_counts['frozen_parameters']:,}")
print(f"Trainable percentage: {param_counts['trainable_parameters']/param_counts['total_parameters']*100:.4f}%")

## Accessing Trainable Parameters

In LoRA fine-tuning, only the LoRA parameters and task-specific head are trainable:

In [ ]:
# Get the parameters that will be trained
trainable_params = model.get_trainable_parameters()
print(f"Number of trainable parameter tensors: {len(trainable_params)}")

# Show the shapes of trainable parameters
for i, param in enumerate(trainable_params):
    print(f"Param {i+1}: {param.shape}")

## Setting Up Optimizer for LoRA Fine-tuning

Since only a small subset of parameters are trainable, we can use a higher learning rate:

In [ ]:
import torch.optim as optim

# Setup optimizer for only the trainable parameters
optimizer = optim.AdamW(
    model.get_trainable_parameters(),  # Only train LoRA and task-specific parameters
    lr=1e-3,                         # Higher LR is often possible with LoRA
    weight_decay=0.01
)

print(f"Optimizer will update {len(model.get_trainable_parameters())} parameter tensors")

## Example Forward Pass

Here's how to perform a forward pass with the LoRA-enhanced model:

In [ ]:
# Create dummy input data (in practice, you would load your actual data)
batch_size = 4
seq_len = 100
num_chromosomes = 23  # Human chromosomes

# Input tensors
value = torch.randn(batch_size, seq_len, 1).to(device)  # Gene expression values
chromosome = torch.randint(0, num_chromosomes, (batch_size, seq_len)).to(device)  # Chromosome indices
pos_start = torch.randint(0, 1000000, (batch_size, seq_len)).to(device)  # Start positions
pos_end = pos_start + torch.randint(100, 1000, (batch_size, seq_len)).to(device)  # End positions

# Forward pass
model.eval()  # Set to evaluation mode for inference
with torch.no_grad():  # Disable gradient computation for inference
    outputs = model(value, chromosome, pos_start, pos_end)
    
print(f"Output shape: {outputs.shape}")
print(f"Output device: {outputs.device}")

## Saving the Fine-tuned Model

After fine-tuning, you can save the model with its LoRA weights:

In [ ]:
import os

# Create output directory
output_dir = "./fine_tuned_models"
os.makedirs(output_dir, exist_ok=True)

# Save the fine-tuned model
output_path = os.path.join(output_dir, "chromfound_lora_finetuned.pt")
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'lora_rank': 8,
        'lora_alpha': 16.0,
        'lora_dropout': 0.0,
        'task_type': 'classification',
        'num_classes': 5,
        'model_args': model.pretrain_model_args
    }
}, output_path)

print(f"Model saved to {output_path}")

## Key Benefits of Using LoRA with ChromFound

1. **Memory Efficiency**: Only a small fraction of parameters are trained, reducing GPU memory requirements
2. **Faster Training**: Fewer parameters to update means faster training iterations
3. **Reduced Overfitting Risk**: With fewer trainable parameters, there's less risk of overfitting on small datasets
4. **Preserved Pre-trained Knowledge**: The original model weights remain unchanged, preserving learned representations
5. **Multi-task Flexibility**: Different LoRA adapters can be trained for different tasks and swapped as needed

This implementation allows you to efficiently adapt ChromFound to your specific single-cell genomics tasks while maintaining the benefits of the pre-trained foundation model.